In [1]:
import pandas as pd
import numpy as np

# Load just 100K rows first — faster for exploration
df = pd.read_csv('../data/raw/PS_20174392719_1491204439457_log.csv', nrows=100000)

# Shape — how many rows, how many columns
print("Shape:", df.shape)

# Column names and data types
print("\nData types:")
print(df.dtypes)

# First 5 rows — see what the data actually looks like
print("\nSample rows:")
print(df.head())

# Check for nulls in each column
print("\nNull counts:")
print(df.isnull().sum())

# What transaction types exist and how common is each
print("\nTransaction type distribution:")
print(df['type'].value_counts())

# What % of transactions are fraudulent
print("\nFraud rate:")
print(df['isFraud'].value_counts(normalize=True) * 100)

# Amount statistics — min, max, mean, percentiles
print("\nAmount statistics:")
print(df['amount'].describe())

# Average amount by transaction type
print("\nAverage amount by type:")
print(df.groupby('type')['amount'].mean().sort_values(ascending=False))

Shape: (100000, 11)

Data types:
step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

Sample rows:
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        

In [1]:
import duckdb

con = duckdb.connect(r'C:\Users\user\Desktop\Datasentinel\datasentinel.db')

In [2]:
result = con.execute("""
    SELECT
        metric_date,
        transaction_type,
        total_transactions,
        LAG(total_transactions) OVER (
            PARTITION BY transaction_type 
            ORDER BY metric_date
        ) AS prev_day_volume,
        ROUND(
            (total_transactions - LAG(total_transactions) OVER (
                PARTITION BY transaction_type 
                ORDER BY metric_date)
            ) * 100.0 / NULLIF(LAG(total_transactions) OVER (
                PARTITION BY transaction_type 
                ORDER BY metric_date), 0)
        , 2) AS day_over_day_pct_change
    FROM aggregated_metrics
    ORDER BY metric_date DESC, transaction_type
""").df()

result

,metric_date,transaction_type,total_transactions,prev_day_volume,day_over_day_pct_change
0,2024-01-01,CASH_IN,99182,<NA>,NaN
1,2024-01-01,CASH_OUT,166056,<NA>,NaN
2,2024-01-01,DEBIT,3276,<NA>,NaN
3,2024-01-01,PAYMENT,150047,<NA>,NaN
4,2024-01-01,TRANSFER,37175,<NA>,NaN


In [3]:
query = """
SELECT
    metric_date,
    transaction_type,
    ROUND(fraud_rate * 100, 4) AS fraud_rate_pct,
    ROUND(AVG(fraud_rate) OVER (
        PARTITION BY transaction_type
        ORDER BY metric_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) * 100, 4) AS fraud_rate_7day_avg,
    ROUND((fraud_rate - AVG(fraud_rate) OVER (
        PARTITION BY transaction_type
        ORDER BY metric_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    )) * 100, 4) AS deviation_from_avg
FROM aggregated_metrics
ORDER BY metric_date DESC
"""

df = con.execute(query).df()
df.head(20)

,metric_date,transaction_type,fraud_rate_pct,fraud_rate_7day_avg,deviation_from_avg
0,2024-01-01,CASH_IN,0.0000,0.0000,0.0
1,2024-01-01,DEBIT,0.0000,0.0000,0.0
2,2024-01-01,PAYMENT,0.0000,0.0000,0.0
3,2024-01-01,CASH_OUT,0.0711,0.0711,0.0
4,2024-01-01,TRANSFER,0.2932,0.2932,0.0


In [4]:
query = """
SELECT
    CAST(DATE '2024-01-01' + INTERVAL (step / 24) DAY AS DATE) AS txn_date,
    transaction_type,
    COUNT(*) AS total,
    SUM(CASE WHEN is_balance_mismatch THEN 1 ELSE 0 END) AS mismatch_count,
    ROUND(
        SUM(CASE WHEN is_balance_mismatch THEN 1 ELSE 0 END) * 100.0
        / COUNT(*), 2
    ) AS mismatch_rate_pct
FROM transformed_transactions
GROUP BY 1,2
ORDER BY 1 DESC, mismatch_rate_pct DESC
"""

df = con.execute(query).df()
df.head(20)

,txn_date,transaction_type,total,mismatch_count,mismatch_rate_pct
0,2024-01-01,CASH_IN,99182,99182.0,100.00
1,2024-01-01,TRANSFER,37175,35235.0,94.78
2,2024-01-01,CASH_OUT,166056,146215.0,88.05
3,2024-01-01,PAYMENT,150047,75258.0,50.16
4,2024-01-01,DEBIT,3276,1024.0,31.26


In [6]:
query = """
SELECT
    CAST(DATE '2024-01-01' + INTERVAL (step / 24) DAY AS DATE) AS txn_date,
    ROUND(
        SUM(
            CASE
                WHEN customer_dest IS NULL THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS customer_dest_null_pct,
    ROUND(
        SUM(
            CASE
                WHEN amount IS NULL THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS amount_null_pct
FROM transformed_transactions
GROUP BY 1
ORDER BY 1 DESC
"""

con.execute(query).df()

,txn_date,customer_dest_null_pct,amount_null_pct
0,2024-01-01,44.95,0.0


In [7]:
query = """
WITH potential_dupes AS (
    SELECT
        transaction_id,
        customer_origin,
        customer_dest,
        amount,
        step,
        COUNT(*) OVER (
            PARTITION BY customer_origin, customer_dest, amount
            ORDER BY step
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ) AS nearby_identical_count
    FROM transformed_transactions
)
SELECT
    COUNT(*) AS total_transactions,
    SUM(
        CASE
            WHEN nearby_identical_count > 1 THEN 1
            ELSE 0
        END
    ) AS potential_duplicates,
    ROUND(
        SUM(
            CASE
                WHEN nearby_identical_count > 1 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        3
    ) AS duplicate_rate_pct
FROM potential_dupes
"""

con.execute(query).df()

,total_transactions,potential_duplicates,duplicate_rate_pct
0,455736,7250.0,1.591


In [8]:
query = """
SELECT
    transaction_type,
    ROUND(
        PERCENTILE_CONT(0.50)
        WITHIN GROUP (ORDER BY amount),
        2
    ) AS median_amount,
    ROUND(
        PERCENTILE_CONT(0.95)
        WITHIN GROUP (ORDER BY amount),
        2
    ) AS p95_amount,
    ROUND(
        PERCENTILE_CONT(0.99)
        WITHIN GROUP (ORDER BY amount),
        2
    ) AS p99_amount,
    ROUND(MAX(amount), 2) AS max_amount,
    ROUND(
        MAX(amount) /
        NULLIF(
            PERCENTILE_CONT(0.95)
            WITHIN GROUP (ORDER BY amount),
            0
        ),
        1
    ) AS max_to_p95_ratio
FROM transformed_transactions
GROUP BY transaction_type
ORDER BY max_to_p95_ratio DESC
"""

con.execute(query).df()

,transaction_type,median_amount,p95_amount,p99_amount,max_amount,max_to_p95_ratio
0,CASH_OUT,159927.35,463277.79,622795.53,1.202972e+09,2596.7
1,CASH_IN,148000.69,428543.15,582077.49,6.351432e+08,1482.1
2,PAYMENT,9327.09,31010.31,44978.68,4.304612e+07,1388.1
3,TRANSFER,525390.01,1970783.71,2922035.90,1.465680e+09,743.7
4,DEBIT,3500.17,17284.26,84704.69,3.635930e+06,210.4


In [9]:
query = """
SELECT
    run_timestamp,
    pipeline_version,
    rows_ingested,
    rows_processed,
    rows_dropped,
    ROUND(
        rows_dropped * 100.0 /
        NULLIF(rows_ingested, 0),
        2
    ) AS drop_rate_pct,
    status,
    error_message
FROM pipeline_run_logs
ORDER BY run_timestamp DESC
"""

con.execute(query).df()

,run_timestamp,pipeline_version,rows_ingested,rows_processed,rows_dropped,drop_rate_pct,status,error_message
0,2026-06-15 14:57:51.524170,v1.0.1,71200,34521,36679,51.52,PARTIAL,None
1,2026-06-14 12:57:51.524170,v1.0.1,71136,71136,21,0.03,SUCCESS,None
2,2026-06-13 12:57:51.524170,v1.0.1,71337,71337,21,0.03,SUCCESS,None
3,2026-06-12 12:57:51.524170,v1.0.0,71477,71477,10,0.01,SUCCESS,None
4,2026-06-11 12:57:51.524170,v1.0.0,71604,71604,19,0.03,SUCCESS,None
5,2026-06-10 12:57:51.524170,v1.0.0,70759,70759,11,0.02,SUCCESS,None


In [10]:
query = """
SELECT
    MAX(step) AS latest_step_in_db,
    744 AS expected_max_step,
    744 - MAX(step) AS missing_steps,
    744 - MAX(step) AS estimated_hours_stale
FROM transformed_transactions
"""

con.execute(query).df()

,latest_step_in_db,expected_max_step,missing_steps,estimated_hours_stale
0,19,744,725,725


In [11]:
query = """
SELECT
    COUNT(*) AS total_transactions,
    SUM(
        CASE
            WHEN customer_dest LIKE 'MERCHANT_DELETED_%'
            THEN 1
            ELSE 0
        END
    ) AS orphaned_refs,
    SUM(
        CASE
            WHEN customer_dest IS NULL
            THEN 1
            ELSE 0
        END
    ) AS null_refs,
    ROUND(
        (
            SUM(
                CASE
                    WHEN customer_dest LIKE 'MERCHANT_DELETED_%'
                    THEN 1
                    ELSE 0
                END
            )
            +
            SUM(
                CASE
                    WHEN customer_dest IS NULL
                    THEN 1
                    ELSE 0
                END
            )
        ) * 100.0 / COUNT(*),
        3
    ) AS integrity_failure_pct
FROM transformed_transactions
"""

con.execute(query).df()

,total_transactions,orphaned_refs,null_refs,integrity_failure_pct
0,455736,479.0,204846.0,45.053


In [12]:
query = """
WITH customer_cohorts AS (
    SELECT
        customer_origin,
        MIN(step / 24) AS cohort_day
    FROM transformed_transactions
    GROUP BY customer_origin
)
SELECT
    CAST(c.cohort_day AS INTEGER) AS cohort_day,
    COUNT(DISTINCT t.customer_origin) AS cohort_size,
    COUNT(*) AS total_transactions,
    SUM(
        CASE
            WHEN t.is_fraud = 0
            THEN 1
            ELSE 0
        END
    ) AS clean_transactions,
    ROUND(
        SUM(
            CASE
                WHEN t.is_fraud = 0
                THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS clean_rate_pct
FROM transformed_transactions t
JOIN customer_cohorts c
    ON t.customer_origin = c.customer_origin
GROUP BY 1
ORDER BY 1
"""

con.execute(query).df()

,cohort_day,cohort_size,total_transactions,clean_transactions,clean_rate_pct
0,0,173417,174873,174735.0,99.92
1,1,278640,280863,280774.0,99.97


In [13]:
con.close()

In [2]:
import duckdb

DB_PATH = r"C:\Users\user\Desktop\Datasentinel\datasentinel.db"
con = duckdb.connect(DB_PATH)

checks = {
    "Raw row count (expect 500,000)":
        "SELECT COUNT(*) FROM raw_transactions",
        
    "Transformed row count (expect ~476,000 — 4.8% less + dupes)":
        "SELECT COUNT(*) FROM transformed_transactions",
        
    "Null customer_dest % (expect ~18% — schema drift)":
        """SELECT ROUND(
               SUM(CASE WHEN customer_dest IS NULL THEN 1 ELSE 0 END) 
               * 100.0 / COUNT(*), 1)
           FROM transformed_transactions""",
           
    "Max-to-P95 ratio (expect >100 — value anomaly)":
        """SELECT ROUND(
               MAX(amount) / PERCENTILE_CONT(0.95) 
               WITHIN GROUP (ORDER BY amount), 1)
           FROM transformed_transactions""",
           
    "Orphaned merchant refs (expect 500)":
        """SELECT COUNT(*) FROM transformed_transactions
           WHERE customer_dest LIKE 'MERCHANT_DELETED_%'""",
           
    "PARTIAL pipeline run (expect 1)":
        """SELECT COUNT(*) FROM pipeline_run_logs 
           WHERE status = 'PARTIAL'""",
           
    "Freshness gap in hours (expect 48)":
        "SELECT 744 - MAX(step) FROM transformed_transactions"
}

print("=" * 50)
print("DataSentinel — Weekend 1 Validation")
print("=" * 50)

all_passed = True
for check_name, query in checks.items():
    result = con.execute(query).fetchone()[0]
    print(f"\n✅ {check_name}")
    print(f"   Result: {result}")

con.close()
print("\n" + "=" * 50)
print("All checks complete. Weekend 1 done ✓")
print("=" * 50)

DataSentinel — Weekend 1 Validation

✅ Raw row count (expect 500,000)
   Result: 500000

✅ Transformed row count (expect ~476,000 — 4.8% less + dupes)
   Result: 479020

✅ Null customer_dest % (expect ~18% — schema drift)
   Result: 9.4

✅ Max-to-P95 ratio (expect >100 — value anomaly)
   Result: 9364.5

✅ Orphaned merchant refs (expect 500)
   Result: 500

✅ PARTIAL pipeline run (expect 1)
   Result: 1

✅ Freshness gap in hours (expect 48)
   Result: 50

All checks complete. Weekend 1 done ✓


In [1]:
import duckdb
con = duckdb.connect(r'C:\Users\user\Desktop\Datasentinel\datasentinel.db')

result = con.execute("SELECT MIN(step), MAX(step), COUNT(*) FROM transformed_transactions").df()
print(result)

result2 = con.execute("SELECT MIN(step), MAX(step), COUNT(*) FROM raw_transactions").df()
print(result2)

   min(step)  max(step)  count_star()
0          1        694        479020
   min(step)  max(step)  count_star()
0          1        742        500000


In [6]:
import os
print(os.getcwd())

c:\Users\user\Desktop\Datasentinel\Notebooks


In [7]:
import sys
sys.path.append("..")

In [8]:
from Detection.statistical import detect_value_anomalies

In [15]:
import os
from dotenv import load_dotenv
load_dotenv(r'C:\Users\user\Desktop\Datasentinel\.env')

import google.generativeai as genai
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_21676\1724135190.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [7]:
import os
from dotenv import load_dotenv
load_dotenv(r'C:\Users\user\Desktop\Datasentinel\.env')

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

response = llm.invoke("Say hello in one short sentence.")
print(response.content)

Hello!


In [ ]:
import sys
sys.path.append(r'C:\Users\user\Desktop\Datasentinel')
from agents.root_cause import investigate_alert

alert_text = """
Issue type: schema_drift
Severity: critical
The transformed_transactions table shows a high null rate
in the customer_dest column, much higher than expected.
"""

answer = investigate_alert(alert_text)
print(answer)

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)




> Entering new AgentExecutor chain...
Action: check_upstream_table
Action Input: {"table_name": "transformed_transactions", "column_name": "customer_dest"}Invalid table. Choose from: ['raw_transactions', 'transformed_transactions']Action: check_upstream_table
Action Input: {"table_name": "transformed_transactions"}Invalid table. Choose from: ['raw_transactions', 'transformed_transactions']Action: check_upstream_table
Action Input: {"table_name": "transformed_transactions"}Invalid table. Choose from: ['raw_transactions', 'transformed_transactions']Action: check_upstream_table
Action Input: {"table_name": "transformed_transactions"}Invalid table. Choose from: ['raw_transactions', 'transformed_transactions']Action: check_upstream_table
Action Input: {"table_name": "transformed_transactions"}Invalid table. Choose from: ['raw_transactions', 'transformed_transactions']Action: check_upstream_table
Action Input: {"table_name": "transformed_transactions"}Invalid table. Choose from: ['raw_tran

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 2.74290293s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_de

Action: check_upstream_table
Action Input: {"table_name": "transformed_transactions"}Invalid table. Choose from: ['raw_transactions', 'transformed_transactions']

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 54.762915166s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_

Action: check_upstream_table
Action Input: {"table_name": "transformed_transactions"}Invalid table. Choose from: ['raw_transactions', 'transformed_transactions']

> Finished chain.
Agent stopped due to iteration limit or time limit.


: 

In [1]:
import sys
sys.path.append(r'C:\Users\user\Desktop\Datasentinel')
from rag.retriever import retrieve_context

query = "schema drift null values customer_dest upstream API change"
context = retrieve_context(query, k=2)
print(context)

Loading FAISS index from disk...
Loading embedding model...


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding model loaded.
FAISS index loaded.
[Source: incident_history.txt]
INCIDENT INC-2023-089 | Severity: P1 | Resolved
Date: September 14, 2023
Issue type: Schema drift — customer_dest null rate spiked to 38%
Root cause: The merchant onboarding API (v2.4) dropped the 
destination_merchant_id field from its response payload after an 
unannounced upstream schema change. The payment ingestion pipeline 
did not validate the field presence before writing to the transformed 
layer, causing all transactions processed after 02:14 AM to have 
null customer_dest values.
Resolution: Pipeline was rolled back to API v2.3 within 3 hours. 
A schema validation step was added at ingestion to check for 
required field presence before committing a batch. 
Business impact: 47,832 transactions affected. Merchant attribution 
reports were unreliable for a 6-hour window. No revenue impact as 
transactions themselves were valid.
Lesson learned: Always validate upstream API response schema on 
every ingest